In [1]:
!pip install -q langchain langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [3]:
import os
import gradio as gr

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

In [5]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

# ---------------------------------------------------------------------------
# Model  (this was missing — caused the NameError)
# ---------------------------------------------------------------------------
model = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

# ---------------------------------------------------------------------------
# Summary Prompt
# ---------------------------------------------------------------------------
summary_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie critic."),
        ("human", "Provide a brief summary of the movie {movie_name}."),
    ]
)

# ---------------------------------------------------------------------------
# Plot Analysis Prompt
# ---------------------------------------------------------------------------
def analyze_plot(plot):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a movie critic."),
            (
                "human",
                "Analyze the plot:\n\n{plot}\n\nWhat are its strengths and weaknesses?",
            ),
        ]
    )
    return prompt.format_prompt(plot=plot)

# ---------------------------------------------------------------------------
# Character Analysis Prompt
# ---------------------------------------------------------------------------
def analyze_characters(characters):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a movie critic."),
            (
                "human",
                "Analyze the characters:\n\n{characters}\n\nWhat are their strengths and weaknesses?",
            ),
        ]
    )
    return prompt.format_prompt(characters=characters)

# ---------------------------------------------------------------------------
# Combine Results
# ---------------------------------------------------------------------------
def combine_verdicts(plot_analysis, character_analysis):
    return f"""
## Plot Analysis

{plot_analysis}

---

## Character Analysis

{character_analysis}
"""

# ---------------------------------------------------------------------------
# Branches
# ---------------------------------------------------------------------------
plot_branch_chain = (
    RunnableLambda(analyze_plot)
    | model
    | StrOutputParser()
)

character_branch_chain = (
    RunnableLambda(analyze_characters)
    | model
    | StrOutputParser()
)

# ---------------------------------------------------------------------------
# Main Chain
# ---------------------------------------------------------------------------
chain = (
    summary_template
    | model
    | StrOutputParser()
    | RunnableParallel(
        branches={
            "plot": plot_branch_chain,
            "characters": character_branch_chain,
        }
    )
    | RunnableLambda(
        lambda x: combine_verdicts(
            x["branches"]["plot"],
            x["branches"]["characters"],
        )
    )
)

# ---------------------------------------------------------------------------
# Quick test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    result = chain.invoke({"movie_name": "Inception"})
    print(result)


## Plot Analysis

**Inception (2010) – Critical Assessment of Its Plot**

---

## 1. Strengths  

| Aspect | Why It Works | Example / Effect |
|--------|--------------|-------------------|
| **Original Concept & High‑Concept Premise** | The idea of “dream‑heists” turns a heist film inside‑out, giving the audience a fresh set of rules (time dilation, layered reality, shared dreaming). | The “kick” that synchronises multiple dream levels creates a ticking‑clock that is both narrative and visual. |
| **Structural Architecture** | Nolan treats the screenplay like a meticulously engineered building: each dream layer is a floor, each floor supports the one above. The parallel editing of the four levels in the climax is a masterclass in temporal choreography. | The famous “spinning top” sequence simultaneously shows the van falling, the hallway fight, the snow‑fortress assault, and the limbo chase—all cut together to maintain a single, coherent tension curve. |
| **Thematic Depth** | The plo

In [6]:
def movie_review(movie_name):
    try:
        return chain.invoke({"movie_name": movie_name})
    except Exception as e:
        return str(e)
demo = gr.Interface(
    fn=movie_review,
    inputs=gr.Textbox(
        label="Movie Name",
        placeholder="Enter a movie name..."
    ),
    outputs=gr.Markdown(label="Movie Analysis"),
    title="🎬 Movie Critic using LangChain",
    description="Enter a movie name to generate a summary, plot analysis, and character analysis."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://174fc29150b7445239.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
